In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("cbb_data.csv")

# Clean column headers (fixes hidden newlines/spaces)
df.columns = (
    df.columns
      .str.replace(r"\s+", " ", regex=True)
      .str.strip()
)

# Drop Unnamed columns safely
df = df.loc[:, ~df.columns.str.startswith("Unnamed:")].copy()

In [3]:
# Keep signed spread for favorite filtering
df["CLOSING SPREAD"] = pd.to_numeric(df["CLOSING SPREAD"], errors="coerce")

# Use ABS spread ONLY for bucketing
df["SPREAD_ABS"] = df["CLOSING SPREAD"].abs()

# Points + outcome columns
df["F"] = pd.to_numeric(df["F"], errors="coerce")
df["Team W/L (1/0)"] = pd.to_numeric(df["Team W/L (1/0)"], errors="coerce")
df["Team ML + Opp Team Over Win/Loss (1/0)"] = pd.to_numeric(
    df["Team ML + Opp Team Over Win/Loss (1/0)"], errors="coerce"
)

print("Min/Max closing spread:", df["CLOSING SPREAD"].min(), df["CLOSING SPREAD"].max())
print("Favorite rows (spread < 0):", (df["CLOSING SPREAD"] < 0).sum())

Min/Max closing spread: -49.5 49.5
Favorite rows (spread < 0): 5761


In [4]:
df.head()

,BIGDATABALL DATASET,GAME-ID,DATE,TEAM,F,OPENING TOTAL,CLOSING SPREAD,CLOSING TOTAL,CLOSING MONEYLINE,PROJ TEAM TOTAL (CLOSE),Team W/L (1/0),Team Over/Under Proj Team Total (1/0),Team ML + Opp Team Over Win/Loss (1/0),SPREAD_ABS
0,NCAAB 2024-2025 Regular Season,401706962,11/4/2024,Howard Bison,57,152.5,28.5,156.5,2200,64,0,0,0.0,28.5
1,NCAAB 2024-2025 Regular Season,401706962,11/4/2024,Kansas Jayhawks,87,152.5,-28.5,156.5,-8000,92.5,1,0,0.0,28.5
2,NCAAB 2024-2025 Regular Season,401719228,11/4/2024,UNC Asheville Bulldogs,54,162.5,28.5,163.5,3000,67.5,0,0,0.0,28.5
3,NCAAB 2024-2025 Regular Season,401719228,11/4/2024,Alabama Crimson Tide,110,162.5,-28.5,163.5,-10000,96,1,1,0.0,28.5
4,NCAAB 2024-2025 Regular Season,401706926,11/4/2024,Jackson State Tigers,40,136.5,36.5,136.5,2000,50,0,0,0.0,36.5


In [5]:
spread_col = "SPREAD_ABS"
points_col = "F"

# Use the absolute spread for bucketing; keep `CLOSING SPREAD` signed for favorite filtering
df[spread_col] = pd.to_numeric(df[spread_col], errors="coerce")
df["points_bucket"] = pd.qcut(df[points_col].rank(method="first"), 3, labels=["low","mid","high"])

In [6]:
spread_bins = [1, 5, 10, np.inf]
spread_labels = ["1–5", "5–10", "10+"]

df["spread_bucket"] = pd.cut(
    df[spread_col] ,
    bins=spread_bins,
    labels=spread_labels,
    right=False
)

In [7]:
df["points_bucket"] = pd.qcut(
    df[points_col].rank(method="first"),
    q=3,
    labels=["low", "mid", "high"]
)

In [8]:
bucket_counts = (
    df
    .groupby(["spread_bucket", "points_bucket"])
    .size()
    .unstack(fill_value=0)
)

print(bucket_counts)

points_bucket   low   mid  high
spread_bucket                  
1–5            1414  1786  1304
5–10           1174  1342  1165
10+            1165   940  1234


In [9]:
cuts, bins = pd.qcut(
    df["F"],
    q=3,
    retbins=True
)

print(bins)

[ 19.  67.  78. 143.]


Now checking to see if the favorites have won (given favourites have won what is the probability that the underdogs score the over)


In [10]:
df["CLOSING MONEYLINE"] = pd.to_numeric(df["CLOSING MONEYLINE"], errors="coerce")
df["Team W/L (1/0)"] = pd.to_numeric(df["Team W/L (1/0)"], errors="coerce")

df_fav = df[df["CLOSING MONEYLINE"] < 0].copy()

p_fav_win = df_fav["Team W/L (1/0)"].mean()
n_games = len(df_fav)

print(f"Probability favorite wins: {p_fav_win:.4f} (n={n_games})")

Probability favorite wins: 0.7253 (n=5988)


In [11]:
df_fav_win = df_fav[df_fav["Team W/L (1/0)"] == 1].copy()

# Ensure the parlay indicator is numeric
df_fav["Team ML + Opp Team Over Win/Loss (1/0)"] = pd.to_numeric(
    df_fav["Team ML + Opp Team Over Win/Loss (1/0)"], errors="coerce"
)

# Joint probability: favorite won AND opponent went over
p_joint = df_fav["Team ML + Opp Team Over Win/Loss (1/0)"].mean()

# Conditional probability: P(opponent over | favorite wins) = P(fav win AND opp over) / P(fav win)
p_opp_over_given_fav_win = p_joint / p_fav_win

n_cond = len(df_fav_win)
print(f"P(opp over | favorite wins) = {p_opp_over_given_fav_win:.4f} (n where fav won={n_cond})")

P(opp over | favorite wins) = 0.3990 (n where fav won=4343)


This is overall now for each bucket 

In [12]:
# ensure numeric (safe even if already done)
df_fav["Team W/L (1/0)"] = pd.to_numeric(df_fav["Team W/L (1/0)"], errors="coerce")
df_fav["Team ML + Opp Team Over Win/Loss (1/0)"] = pd.to_numeric(
    df_fav["Team ML + Opp Team Over Win/Loss (1/0)"], errors="coerce"
)

# P(Favorite wins) per bucket
p_fav_win_bucket = (
    df_fav
    .groupby(["spread_bucket", "points_bucket"])["Team W/L (1/0)"]
    .mean()
)

# P(Favorite wins AND Opponent over) per bucket
p_joint_bucket = (
    df_fav
    .groupby(["spread_bucket", "points_bucket"])["Team ML + Opp Team Over Win/Loss (1/0)"]
    .mean()
)

# Conditional probability per bucket
p_opp_over_given_fav_win = (p_joint_bucket / p_fav_win_bucket).unstack()

# Sample size: games where favorite WON in each bucket
n_fav_win_bucket = (
    df_fav[df_fav["Team W/L (1/0)"] == 1]
    .groupby(["spread_bucket", "points_bucket"])
    .size()
    .unstack(fill_value=0)
)

# Pretty table: prob (n)
result = (
    p_opp_over_given_fav_win.round(3).astype(str)
    + " ("
    + n_fav_win_bucket.astype(str)
    + ")"
)

print("\nP(Underdog Over | Favorite Wins) by bucket:")
print(result)


P(Underdog Over | Favorite Wins) by bucket:
points_bucket          low          mid          high
spread_bucket                                        
1–5            0.018 (221)  0.215 (583)   0.609 (609)
5–10           0.026 (155)  0.223 (512)   0.564 (713)
10+            0.121 (116)  0.375 (405)  0.531 (1027)


In [13]:
df_b = df[df["CLOSING SPREAD"] < 0].copy()
df_b = df_b.dropna(subset=["spread_bucket"])

df_b["Team W/L (1/0)"] = pd.to_numeric(df_b["Team W/L (1/0)"], errors="coerce")
df_b["Team ML + Opp Team Over Win/Loss (1/0)"] = pd.to_numeric(
    df_b["Team ML + Opp Team Over Win/Loss (1/0)"], errors="coerce"
)

b = (df_b["spread_bucket"]=="1–5") & (df_b["points_bucket"]=="low")

p_fav = df_b.loc[b, "Team W/L (1/0)"].mean()
p_joint = df_b.loc[b, "Team ML + Opp Team Over Win/Loss (1/0)"].mean()
ratio = p_joint / p_fav if (p_fav and not np.isnan(p_fav)) else np.nan

print("p_fav:", p_fav)
print("p_joint:", p_joint)
print("ratio:", ratio)
print("n rows in bucket:", b.sum())
print("n fav wins in bucket:", (df_b.loc[b, "Team W/L (1/0)"] == 1).sum())

p_fav: 0.3312
p_joint: 0.00641025641025641
ratio: 0.01935463891985631
n rows in bucket: 625
n fav wins in bucket: 207


In [14]:
# 1) how many favorite rows exist BEFORE dropping NaNs?
df_b0 = df[df["CLOSING SPREAD"] < 0].copy()
print("Favorite rows before dropna:", len(df_b0))

# 2) do the bucket columns exist?
print("Has spread_bucket:", "spread_bucket" in df_b0.columns)
print("Has points_bucket:", "points_bucket" in df_b0.columns)

# 3) if they exist, what's the NaN rate?
if "spread_bucket" in df_b0.columns and "points_bucket" in df_b0.columns:
    print("spread_bucket NA rate:", df_b0["spread_bucket"].isna().mean())
    print("points_bucket NA rate:", df_b0["points_bucket"].isna().mean())

# 4) show a few spreads to confirm the filter is sane
print(df_b0[["CLOSING SPREAD"]].head(10))

Favorite rows before dropna: 5761
Has spread_bucket: True
Has points_bucket: True
spread_bucket NA rate: 0.0
points_bucket NA rate: 0.0
    CLOSING SPREAD
1            -28.5
3            -28.5
5            -36.5
7            -46.5
9             -6.5
11           -32.5
13           -27.5
15           -36.5
17           -29.5
18            -6.5


In [15]:
df_bucket = df_b.dropna(subset=["spread_bucket","points_bucket"]).copy()

fav_wins = (
    df_bucket[df_bucket["Team W/L (1/0)"] == 1]
    .groupby(["spread_bucket","points_bucket"])
    .size()
    .unstack(fill_value=0)
)

fav_win_opp_over = (
    df_bucket[df_bucket["Team ML + Opp Team Over Win/Loss (1/0)"] == 1]
    .groupby(["spread_bucket","points_bucket"])
    .size()
    .unstack(fill_value=0)
)

print("Fav wins (denominator):")
print(fav_wins)

print("\nFav win + opp over (numerator):")
print(fav_win_opp_over)

print("\nConditional probability:")
print((fav_win_opp_over / fav_wins).replace([np.inf, -np.inf], np.nan))

Fav wins (denominator):
points_bucket  low  mid  high
spread_bucket                
1–5            207  541   556
5–10           155  512   713
10+            116  405  1028

Fav win + opp over (numerator):
points_bucket  low  mid  high
spread_bucket                
1–5              4  121   342
5–10             4  114   402
10+             14  152   546

Conditional probability:
points_bucket       low       mid      high
spread_bucket                              
1–5            0.019324  0.223660  0.615108
5–10           0.025806  0.222656  0.563815
10+            0.120690  0.375309  0.531128


In [16]:
df_b = df_fav.dropna(subset=["spread_bucket","points_bucket"]).copy()

df_b["Team Over/Under Proj Team Total (1/0)"] = pd.to_numeric(
    df_b["Team Over/Under Proj Team Total (1/0)"], errors="coerce"
)

# Under-dog wins = favorite losses
df_udog_win_b = df_b[df_b["Team W/L (1/0)"] == 0].copy()

prob_bucket = (
    df_udog_win_b
    .groupby(["spread_bucket","points_bucket"])["Team Over/Under Proj Team Total (1/0)"]
    .mean()
    .unstack()
)

n_bucket = (
    df_udog_win_b
    .groupby(["spread_bucket","points_bucket"])
    .size()
    .unstack(fill_value=0)
)

result = prob_bucket.round(3).astype(str) + " (" + n_bucket.astype(str) + ")"
print(result)

points_bucket          low          mid         high
spread_bucket                                       
1–5            0.013 (466)  0.336 (432)  0.904 (167)
5–10           0.005 (191)   0.24 (192)   0.792 (77)
10+               0.0 (42)   0.038 (53)    0.68 (25)


In [17]:
print("this is a sanity check, both numbers should be the same")
print("Total underdog wins:", (df_fav["Team W/L (1/0)"] == 0).sum())
print("Sum of bucket underdog wins:", n_bucket.values.sum())

this is a sanity check, both numbers should be the same
Total underdog wins: 1645
Sum of bucket underdog wins: 1645


In [18]:
pd.set_option("display.max_rows", 300)   # or None for all rows
pd.set_option("display.max_columns", None)

# Favorites only
df_fav = df[df["CLOSING SPREAD"] < 0].copy()

# Filter: spread bucket 1–5, points low, favorite wins
mask = (
    (df_fav["spread_bucket"] == "1–5") &
    (df_fav["points_bucket"] == "low") &
    (df_fav["Team W/L (1/0)"] == 1)
)

# Show the column headers (all column names)
print(df_fav.columns.tolist())

# Show all matching rows (or cap with head if you want)
display(df_fav.loc[mask])          # shows all (Jupyter may still preview)
print("Rows matched:", mask.sum())

# If you specifically want the first 207 rows:
# display(df_fav.loc[mask].head(207))

['BIGDATABALL DATASET', 'GAME-ID', 'DATE', 'TEAM', 'F', 'OPENING TOTAL', 'CLOSING SPREAD', 'CLOSING TOTAL', 'CLOSING MONEYLINE', 'PROJ TEAM TOTAL (CLOSE)', 'Team W/L (1/0)', 'Team Over/Under Proj Team Total (1/0)', 'Team ML + Opp Team Over Win/Loss (1/0)', 'SPREAD_ABS', 'points_bucket', 'spread_bucket']


,BIGDATABALL DATASET,GAME-ID,DATE,TEAM,F,OPENING TOTAL,CLOSING SPREAD,CLOSING TOTAL,CLOSING MONEYLINE,PROJ TEAM TOTAL (CLOSE),Team W/L (1/0),Team Over/Under Proj Team Total (1/0),Team ML + Opp Team Over Win/Loss (1/0),SPREAD_ABS,points_bucket,spread_bucket
131,NCAAB 2024-2025 Regular Season,401721074,11/4/2024,Coastal Carolina Chanticleers,60,144.5,-4.5,140.5,-215.0,72.5,1,0,0.0,4.5,low,1–5
191,NCAAB 2024-2025 Regular Season,401700387,11/4/2024,La Salle Explorers,65,136.5,-4.5,136.5,-220.0,70.5,1,0,0.0,4.5,low,1–5
255,NCAAB 2024-2025 Regular Season,401721762,11/4/2024,Army Black Knights,67,143.5,-1.5,143.5,-130.0,72.5,1,0,0.0,1.5,low,1–5
655,NCAAB 2024-2025 Regular Season,401711703,11/8/2024,Queens University Royals,67,157.5,-4.5,159.0,-192.0,81.75,1,0,0.0,4.5,low,1–5
724,NCAAB 2024-2025 Regular Season,401720978,11/8/2024,UC Irvine Anteaters,66,150.5,-1.5,151.0,-125.0,76.25,1,0,0.0,1.5,low,1–5
881,NCAAB 2024-2025 Regular Season,401727833,11/10/2024,Drake Bulldogs,66,142.5,-4.0,139.5,-192.0,71.75,1,0,0.0,4.0,low,1–5
1046,NCAAB 2024-2025 Regular Season,401722095,11/12/2024,Fairfield Stags,62,151.5,-1.5,152.5,-125.0,77,1,0,0.0,1.5,low,1–5
1209,NCAAB 2024-2025 Regular Season,401718458,11/13/2024,Charlotte 49ers,65,142.5,-1.5,141.5,-125.0,71.5,1,0,0.0,1.5,low,1–5
1515,NCAAB 2024-2025 Regular Season,401715390,11/16/2024,Minnesota Golden Gophers,59,138.5,-4.5,138.5,-205.0,71.5,1,0,0.0,4.5,low,1–5
1563,NCAAB 2024-2025 Regular Season,401726049,11/16/2024,Towson Tigers,67,138.5,-2.0,136.5,-130.0,69.25,1,0,0.0,2.0,low,1–5


Rows matched: 207


In [19]:
# Favorite rows matching your condition
df_fav = df[df["CLOSING SPREAD"] < 0].copy()

mask = (
    (df_fav["spread_bucket"] == "1–5") &
    (df_fav["points_bucket"] == "low") &
    (df_fav["Team W/L (1/0)"] == 1)
)

game_ids = df_fav.loc[mask, "GAME-ID"].unique()

# Underdog rows from those same games
df_udog_same_games = df[(df["CLOSING SPREAD"] > 0) & (df["GAME-ID"].isin(game_ids))].copy()

display(df_udog_same_games)
print("Underdog rows (same games):", len(df_udog_same_games))

,BIGDATABALL DATASET,GAME-ID,DATE,TEAM,F,OPENING TOTAL,CLOSING SPREAD,CLOSING TOTAL,CLOSING MONEYLINE,PROJ TEAM TOTAL (CLOSE),Team W/L (1/0),Team Over/Under Proj Team Total (1/0),Team ML + Opp Team Over Win/Loss (1/0),SPREAD_ABS,points_bucket,spread_bucket
130,NCAAB 2024-2025 Regular Season,401721074,11/4/2024,Western Michigan Broncos,56,144.5,4.5,140.5,165.0,68,0,0,0.0,4.5,low,1–5
190,NCAAB 2024-2025 Regular Season,401700387,11/4/2024,American University Eagles,52,136.5,4.5,136.5,170.0,66,0,0,0.0,4.5,low,1–5
254,NCAAB 2024-2025 Regular Season,401721762,11/4/2024,UAlbany Great Danes,59,143.5,1.5,143.5,NaN,71,0,0,0.0,1.5,low,1–5
654,NCAAB 2024-2025 Regular Season,401711703,11/8/2024,Western Carolina Catamounts,54,157.5,4.5,159.0,152.0,77.25,0,0,0.0,4.5,low,1–5
725,NCAAB 2024-2025 Regular Season,401720978,11/8/2024,Loyola Marymount Lions,51,150.5,1.5,151.0,105.0,74.75,0,0,0.0,1.5,low,1–5
880,NCAAB 2024-2025 Regular Season,401727833,11/10/2024,Stephen F. Austin Lumberjacks,51,142.5,4.0,139.5,160.0,67.75,0,0,0.0,4.0,low,1–5
1047,NCAAB 2024-2025 Regular Season,401722095,11/12/2024,New Hampshire Wildcats,56,151.5,1.5,152.5,105.0,75.5,0,0,0.0,1.5,low,1–5
1208,NCAAB 2024-2025 Regular Season,401718458,11/13/2024,Richmond Spiders,48,142.5,1.5,141.5,105.0,70,0,0,0.0,1.5,low,1–5
1514,NCAAB 2024-2025 Regular Season,401715390,11/16/2024,Yale Bulldogs,56,138.5,4.5,138.5,170.0,67,0,0,0.0,4.5,low,1–5
1562,NCAAB 2024-2025 Regular Season,401726049,11/16/2024,James Madison Dukes,63,138.5,2.0,136.5,110.0,67.25,0,0,0.0,2.0,low,1–5


Underdog rows (same games): 206


In [20]:
#finding the amount of points that the underdogs scores based ont he projected team totals 

points_col = "PROJ TEAM TOTAL (CLOSE)"

# make numeric
df[points_col] = pd.to_numeric(df[points_col], errors="coerce")

# Favorite row per game
fav_rows = df[df["CLOSING SPREAD"] < 0].copy()

# Underdog row per game: grab underdog projected total
udog_rows = df[df["CLOSING SPREAD"] > 0][["GAME-ID", points_col]].copy()
udog_rows = udog_rows.rename(columns={points_col: "UDOG_PROJ_TT_CLOSE"})

# One row per game (anchored on favorite) + underdog projected total
games = fav_rows.merge(udog_rows, on="GAME-ID", how="left")

print("Games (favorite rows):", len(games))
print("Missing UDOG_PROJ_TT_CLOSE:", games["UDOG_PROJ_TT_CLOSE"].isna().sum())

games[["GAME-ID", "TEAM", "CLOSING SPREAD", "spread_bucket", "UDOG_PROJ_TT_CLOSE"]].head()

Games (favorite rows): 5761
Missing UDOG_PROJ_TT_CLOSE: 5


,GAME-ID,TEAM,CLOSING SPREAD,spread_bucket,UDOG_PROJ_TT_CLOSE
0,401706962,Kansas Jayhawks,-28.5,10+,64.0
1,401719228,Alabama Crimson Tide,-28.5,10+,67.5
2,401706926,Houston Cougars,-36.5,10+,50.0
3,401706954,Iowa State Cyclones,-46.5,10+,43.5
4,401727062,Gonzaga Bulldogs,-6.5,5–10,75.0


In [21]:
games["udog_points_bucket"] = (
    games.groupby("spread_bucket")["UDOG_PROJ_TT_CLOSE"]
         .transform(lambda s: pd.qcut(s.rank(method="first"), 3, labels=["low","mid","high"]))
)

# Your new 3x3 count table (spread_bucket × underdog proj total bucket)
bucket_counts = (
    games.groupby(["spread_bucket", "udog_points_bucket"])
         .size()
         .unstack(fill_value=0)
)

print(bucket_counts)

udog_points_bucket  low  mid  high
spread_bucket                     
1–5                 750  749   749
5–10                613  613   613
10+                 557  556   556


In [22]:
ranges = (
    games.groupby(["spread_bucket","udog_points_bucket"])["UDOG_PROJ_TT_CLOSE"]
         .agg(n="count", min_val="min", max_val="max")
)

print(ranges)

                                    n  min_val  max_val
spread_bucket udog_points_bucket                       
1–5           low                 750    57.25    68.75
              mid                 749    68.75    72.50
              high                749    72.50    88.50
5–10          low                 613    55.00    67.00
              mid                 613    67.00    70.75
              high                613    70.75    86.00
10+           low                 557    43.25    62.25
              mid                 556    62.25    66.75
              high                556    66.75    84.50


In [23]:
ranges = (
    games.groupby(["spread_bucket","udog_points_bucket"])["UDOG_PROJ_TT_CLOSE"]
         .agg(
             n="count",
             min_val="min",
             max_val="max",
             mean_val="mean"
         )
         .round(2)
)

print(ranges)


                                    n  min_val  max_val  mean_val
spread_bucket udog_points_bucket                                 
1–5           low                 750    57.25    68.75     65.98
              mid                 749    68.75    72.50     70.61
              high                749    72.50    88.50     75.74
5–10          low                 613    55.00    67.00     64.29
              mid                 613    67.00    70.75     68.87
              high                613    70.75    86.00     73.71
10+           low                 557    43.25    62.25     58.69
              mid                 556    62.25    66.75     64.55
              high                556    66.75    84.50     70.08


In [25]:
#finding the probabilities again 
games["Team W/L (1/0)"] = pd.to_numeric(games["Team W/L (1/0)"], errors="coerce")
games["Team ML + Opp Team Over Win/Loss (1/0)"] = pd.to_numeric(
    games["Team ML + Opp Team Over Win/Loss (1/0)"], errors="coerce"
)
games["Team Over/Under Proj Team Total (1/0)"] = pd.to_numeric(
    games["Team Over/Under Proj Team Total (1/0)"], errors="coerce"
)
games["Opp Over (1/0)"] = np.where(
    games["Team W/L (1/0)"] == 1,
    games["Team ML + Opp Team Over Win/Loss (1/0)"],
    1 - games["Team Over/Under Proj Team Total (1/0)"]
)

In [26]:
# denominator: favorite wins
fav_wins = (
    games[games["Team W/L (1/0)"] == 1]
    .groupby(["spread_bucket","udog_points_bucket"])
    .size()
)

# numerator: favorite win AND opponent over
joint = (
    games[games["Team ML + Opp Team Over Win/Loss (1/0)"] == 1]
    .groupby(["spread_bucket","udog_points_bucket"])
    .size()
)

p1 = (joint / fav_wins).unstack()

print("P(Underdog Over | Favorite Wins)")
print(p1.round(3))

P(Underdog Over | Favorite Wins)
udog_points_bucket    low    mid   high
spread_bucket                          
1–5                 0.371  0.332  0.372
5–10                0.369  0.372  0.390
10+                 0.491  0.444  0.443


In [27]:
# denominator: opponent over
opp_over = (
    games[games["Opp Over (1/0)"] == 1]
    .groupby(["spread_bucket","udog_points_bucket"])
    .size()
)

p2 = (joint / opp_over).unstack()

print("\nP(Favorite Wins | Underdog Over)")
print(p2.round(3))


P(Favorite Wins | Underdog Over)
udog_points_bucket    low    mid   high
spread_bucket                          
1–5                 0.424  0.356  0.428
5–10                0.571  0.616  0.603
10+                 0.915  0.878  0.831


In [32]:
import pandas as pd
import numpy as np

df["CLOSING SPREAD"] = pd.to_numeric(df["CLOSING SPREAD"], errors="coerce")
df["CLOSING TOTAL"]  = pd.to_numeric(df["CLOSING TOTAL"], errors="coerce")

games = (
    df.groupby("GAME-ID", as_index=False)
      .agg(
          SPREAD_ABS=("CLOSING SPREAD", lambda s: np.nanmax(np.abs(s))),
          CLOSING_TOTAL=("CLOSING TOTAL", "first"),
          DATE=("DATE", "first")  # optional
      )
)

print("Unique games:", len(games))
print("Missing CLOSING_TOTAL:", games["CLOSING_TOTAL"].isna().sum())

Unique games: 6294
Missing CLOSING_TOTAL: 526


/var/folders/tk/phxlpd010yl45ky8mgmjkhlc0000gq/T/ipykernel_36256/3435049647.py:10: RuntimeWarning: All-NaN axis encountered
  SPREAD_ABS=("CLOSING SPREAD", lambda s: np.nanmax(np.abs(s))),


In [34]:
games["spread_bucket"] = pd.cut(
    games["SPREAD_ABS"],
    bins=[1, 5, 10, np.inf],
    labels=["1–5", "5–10", "10+"],
    right=False
)

print(games["spread_bucket"].value_counts(dropna=False))

# Quantile cut points
q1, q2 = games["CLOSING_TOTAL"].quantile([1/3, 2/3]).tolist()

# Round to nearest 0.5 for cleaner sportsbook-style numbers
def round_to_half(x):
    return np.round(x * 2) / 2

c1 = float(round_to_half(q1))
c2 = float(round_to_half(q2))

# Ensure strictly increasing (rare edge case if data is very tied)
if c2 <= c1:
    c2 = c1 + 0.5

bins = [-np.inf, c1, c2, np.inf]
labels = [
    f"{c1:.1f} and lower",
    f"{c1:.1f} to {c2:.1f}",
    f"{c2:.1f}+"
]

games["total_points_bucket"] = pd.cut(
    games["CLOSING_TOTAL"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=True
)

print("Total-points bin edges used:", bins)
print("Bucket labels:", labels)
print("\nCounts per total_points_bucket:")
print(games["total_points_bucket"].value_counts(dropna=False))

print("\nActual ranges per bucket (min/max):")
print(
    games.groupby("total_points_bucket")["CLOSING_TOTAL"]
         .agg(n="count", min_val="min", max_val="max")
         .round(2)
)

spread_bucket
1–5     2255
5–10    1842
10+     1670
NaN      527
Name: count, dtype: int64
Total-points bin edges used: [-inf, 141.0, 148.5, inf]
Bucket labels: ['141.0 and lower', '141.0 to 148.5', '148.5+']

Counts per total_points_bucket:
total_points_bucket
141.0 to 148.5     2007
141.0 and lower    1938
148.5+             1823
NaN                 526
Name: count, dtype: int64

Actual ranges per bucket (min/max):
                        n  min_val  max_val
total_points_bucket                        
141.0 and lower      1938    116.5    141.0
141.0 to 148.5       2007    141.5    148.5
148.5+               1823    149.0    181.5


In [35]:
bucket_counts = (
    games.groupby(["spread_bucket", "total_points_bucket"])
         .size()
         .unstack(fill_value=0)
)

print(bucket_counts)

total_points_bucket  141.0 and lower  141.0 to 148.5  148.5+
spread_bucket                                               
1–5                              841             773     641
5–10                             603             651     588
10+                              494             582     594


In [36]:
# Grab favorite rows (spread < 0)
fav_rows = df[df["CLOSING SPREAD"] < 0][[
    "GAME-ID",
    "Team W/L (1/0)",
    "Team Over/Under Proj Team Total (1/0)",
    "Team ML + Opp Team Over Win/Loss (1/0)"
]].copy()

# Merge onto game-level table
games = games.merge(fav_rows, on="GAME-ID", how="left")

# Ensure numeric
games["Team W/L (1/0)"] = pd.to_numeric(games["Team W/L (1/0)"], errors="coerce")
games["Team Over/Under Proj Team Total (1/0)"] = pd.to_numeric(
    games["Team Over/Under Proj Team Total (1/0)"], errors="coerce"
)
games["Team ML + Opp Team Over Win/Loss (1/0)"] = pd.to_numeric(
    games["Team ML + Opp Team Over Win/Loss (1/0)"], errors="coerce"
)

In [37]:
games["Opp Over (1/0)"] = np.where(
    games["Team W/L (1/0)"] == 1,
    games["Team ML + Opp Team Over Win/Loss (1/0)"],
    1 - games["Team Over/Under Proj Team Total (1/0)"]
)
# Denominator: favorite wins
fav_wins = (
    games[games["Team W/L (1/0)"] == 1]
    .groupby(["spread_bucket", "total_points_bucket"])
    .size()
)

# Numerator: favorite win AND opponent over
joint = (
    games[games["Team ML + Opp Team Over Win/Loss (1/0)"] == 1]
    .groupby(["spread_bucket", "total_points_bucket"])
    .size()
)

p1 = (joint / fav_wins).unstack()

print("P(Underdog Over | Favorite Wins)")
print(p1.round(3))

P(Underdog Over | Favorite Wins)
total_points_bucket  141.0 and lower  141.0 to 148.5  148.5+
spread_bucket                                               
1–5                            0.350           0.349   0.379
5–10                           0.380           0.363   0.389
10+                            0.457           0.463   0.459


In [38]:
# Denominator: opponent over
opp_over = (
    games[games["Opp Over (1/0)"] == 1]
    .groupby(["spread_bucket", "total_points_bucket"])
    .size()
)

p2 = (joint / opp_over).unstack()

print("\nP(Favorite Wins | Underdog Over)")
print(p2.round(3))


P(Favorite Wins | Underdog Over)
total_points_bucket  141.0 and lower  141.0 to 148.5  148.5+
spread_bucket                                               
1–5                            0.400           0.385   0.426
5–10                           0.564           0.615   0.610
10+                            0.871           0.867   0.889
